# 🧠 Deep Learning Architectures: Attention & Object Detection Metrics

In this notebook, we visualize and explore:
1. **Scaled Dot-Product Attention Heatmaps** (interpreting attention distribution matrices).
2. **Non-Maximum Suppression (NMS)** and Intersection over Union (IoU) box filtering.
3. **Residual Block** skip connection verification.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from attention import scaled_dot_product_attention
from detection_metrics import non_max_suppression

print("Deep Learning modules successfully imported!")

### 1. Visualizing Self-Attention Matrices
We generate queries and keys for a sequence of 6 tokens and compute their attention weights.

In [ ]:
np.random.seed(42)
seq_len = 6
d_k = 16

Q = np.random.randn(1, seq_len, d_k)
K = np.random.randn(1, seq_len, d_k)
V = np.random.randn(1, seq_len, d_k)

# Unmasked Attention
_, weights = scaled_dot_product_attention(Q, K, V)

# Causal (Autoregressive) Masked Attention
mask = np.tril(np.ones((seq_len, seq_len)))
_, masked_weights = scaled_dot_product_attention(Q, K, V, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(weights[0], cmap='viridis')
axes[0].set_title("Full Bidirectional Self-Attention")
axes[0].set_xlabel("Key Token Position")
axes[0].set_ylabel("Query Token Position")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(masked_weights[0], cmap='viridis')
axes[1].set_title("Causal (Decoder) Masked Attention")
axes[1].set_xlabel("Key Token Position")
axes[1].set_ylabel("Query Token Position")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

### 2. Object Detection Non-Maximum Suppression (NMS)
Filtering multiple noisy detections down to distinct bounding boxes.

In [ ]:
boxes = np.array([
    [50.0, 50.0, 150.0, 150.0],   # Box A (score 0.92)
    [55.0, 52.0, 152.0, 148.0],   # Box A duplicate (score 0.81) -> Suppressed
    [48.0, 53.0, 149.0, 155.0],   # Box A duplicate (score 0.70) -> Suppressed
    [200.0, 200.0, 300.0, 300.0], # Box B (score 0.88)
    [205.0, 198.0, 302.0, 295.0]  # Box B duplicate (score 0.65) -> Suppressed
])
scores = np.array([0.92, 0.81, 0.70, 0.88, 0.65])

kept_indices = non_max_suppression(boxes, scores, iou_threshold=0.5, score_threshold=0.5)
print(f"Original box count: {len(boxes)}")
print(f"Kept box indices after NMS: {list(kept_indices)}")
print("Preserved boxes successfully isolate the 2 distinct physical objects!")